# 🤖 Notebook 03 — Yield Prediction Model
## Fab Intelligence: Semiconductor Manufacturing Analytics

**Business Question:** *Can we predict whether a wafer will fail BEFORE it reaches final test — using in-line sensor data collected during fabrication?*

**Why it matters:** Each wafer that fails at the end of a 1,500-step process represents $3,000–$19,500 in lost material and machine time. Predicting failure early means pulling wafers before wasting more processing steps.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_auc_score, roc_curve, precision_recall_curve)
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

BLUE = '#1A3C6E'; RED = '#C0392B'; GREEN = '#27AE60'; ORANGE = '#E67E22'; GRAY='#95A5A6'
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'figure.facecolor': '#F8FAFC'})

df = pd.read_csv('../data/secom_combined.csv')
top_sensors = pd.read_csv('../data/top_sensors.csv')
sensor_cols = [c for c in df.columns if c not in ['label', 'label_text']]

# Use top 40 sensors for modeling
top40_sensors = top_sensors.head(40)['sensor'].tolist()
X_raw = df[top40_sensors]
y = (df['label'] == 1).astype(int)

# Preprocess
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X_raw), columns=X_raw.columns)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

print(f'Features: {X_scaled.shape[1]}  |  Failures: {y.sum()} / {len(y)} ({y.mean()*100:.1f}%)')

In [ ]:
# ── Handle Class Imbalance with SMOTE ─────────────────────────────────────
# Real semiconductor data: ~93% pass, ~7% fail — severely imbalanced
print('Applying SMOTE to handle class imbalance (93% PASS vs 7% FAIL)...')
smote = SMOTE(random_state=42, k_neighbors=5)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y)
print(f'After SMOTE — PASS: {(y_resampled==0).sum()}  FAIL: {(y_resampled==1).sum()}')

In [ ]:
# ── Train & Evaluate 3 Models ─────────────────────────────────────────────
models = {
    'Random Forest':      RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
    'Gradient Boosting':  GradientBoostingClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    y_pred_prob = cross_val_predict(model, X_resampled, y_resampled, cv=cv, method='predict_proba')[:, 1]
    y_pred = (y_pred_prob > 0.5).astype(int)
    auc = roc_auc_score(y_resampled, y_pred_prob)
    cm = confusion_matrix(y_resampled, y_pred)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp + 1e-9)
    recall    = tp / (tp + fn + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    results[name] = {'auc': auc, 'precision': precision, 'recall': recall,
                     'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
                     'y_prob': y_pred_prob}
    print(f'{name:<25} AUC: {auc:.3f}  Precision: {precision:.3f}  Recall: {recall:.3f}  F1: {f1:.3f}')

In [ ]:
# ── Figure 3: Model Performance Dashboard ─────────────────────────────────
fig = plt.figure(figsize=(18, 10), facecolor='#F8FAFC')
fig.suptitle('Yield Failure Prediction — Model Performance', 
             fontsize=16, fontweight='bold', color=BLUE)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.4)

model_colors = {'Random Forest': BLUE, 'Gradient Boosting': ORANGE, 'Logistic Regression': GREEN}

# 3a. ROC Curves
ax1 = fig.add_subplot(gs[0, 0:2])
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_resampled, res['y_prob'])
    ax1.plot(fpr, tpr, color=model_colors[name], linewidth=2,
             label=f'{name} (AUC = {res["auc"]:.3f})')
ax1.plot([0,1],[0,1], color=GRAY, linestyle='--', alpha=0.6, label='Random Classifier')
ax1.fill_between([0,1],[0,1], alpha=0.05, color=GRAY)
ax1.set_xlabel('False Positive Rate (Good wafers flagged as bad)')
ax1.set_ylabel('True Positive Rate (Bad wafers caught)')
ax1.set_title('ROC Curve — Wafer Failure Detection', fontweight='bold', color=BLUE)
ax1.legend(loc='lower right', fontsize=9)
ax1.set_facecolor('#FAFCFF')

# 3b. Model comparison bar chart
ax2 = fig.add_subplot(gs[0, 2])
metrics = ['auc', 'precision', 'recall', 'f1']
metric_labels = ['AUC', 'Precision', 'Recall', 'F1']
x = np.arange(len(metrics))
width = 0.25
for i, (name, res) in enumerate(results.items()):
    vals = [res[m] for m in metrics]
    bars = ax2.bar(x + i*width, vals, width, label=name, 
                   color=model_colors[name], alpha=0.85)
ax2.set_xticks(x + width)
ax2.set_xticklabels(metric_labels)
ax2.set_ylim(0, 1.1)
ax2.set_title('Model Comparison', fontweight='bold', color=BLUE)
ax2.legend(fontsize=7)
ax2.set_ylabel('Score')

# 3c. Confusion Matrix for best model (RF)
ax3 = fig.add_subplot(gs[1, 0])
best = results['Random Forest']
cm_data = np.array([[best['tn'], best['fp']], [best['fn'], best['tp']]])
sns.heatmap(cm_data, annot=True, fmt='g', cmap='Blues', ax=ax3,
            xticklabels=['Predicted PASS', 'Predicted FAIL'],
            yticklabels=['Actual PASS', 'Actual FAIL'],
            cbar=False, annot_kws={'size': 13, 'weight': 'bold'})
ax3.set_title('Confusion Matrix\n(Random Forest — Best Model)', fontweight='bold', color=BLUE)

# 3d. Business impact of the model
ax4 = fig.add_subplot(gs[1, 1:])
wafer_cost = 5000  # $5,000/wafer mid-range estimate
monthly_starts = 5000
fail_rate = y.mean()
monthly_fails = monthly_starts * fail_rate

scenarios = {
    'No Model\n(Status Quo)': monthly_fails * wafer_cost,
    'With Logistic\nRegression': monthly_fails * wafer_cost * (1 - results['Logistic Regression']['recall'] * 0.6),
    'With Gradient\nBoosting': monthly_fails * wafer_cost * (1 - results['Gradient Boosting']['recall'] * 0.7),
    'With Random\nForest (Best)': monthly_fails * wafer_cost * (1 - results['Random Forest']['recall'] * 0.75),
}
bar_colors = [RED, ORANGE, ORANGE, GREEN]
bars = ax4.bar(scenarios.keys(), scenarios.values(), color=bar_colors, alpha=0.85, edgecolor='white')
for bar, val in zip(bars, scenarios.values()):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
             f'${val:,.0f}/mo', ha='center', fontsize=9, fontweight='bold')
ax4.set_ylabel('Estimated Monthly Loss ($)')
ax4.set_title('Business Impact: Estimated Monthly Scrap Cost by Model\n(at 5,000 wafers/month, $5,000/wafer)', 
              fontweight='bold', color=BLUE)
savings = scenarios['No Model\n(Status Quo)'] - scenarios['With Random\nForest (Best)']
ax4.annotate(f'Potential savings:\n${savings:,.0f}/month', 
             xy=(3, scenarios['With Random\nForest (Best)']),
             xytext=(2.2, scenarios['No Model\n(Status Quo)'] * 0.85),
             fontsize=10, color=GREEN, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color=GREEN))

plt.savefig('../docs/fig03_model_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 03 saved.')